In [2]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import numpy as np
from evedesign.system import System, Protein
from evedesign.models.boltzfold import BoltzFoldTransformer
from evedesign.utils import ensure_sequence

/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[11:25:40] Initializing Normalizer


In [3]:
seq = "TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH"
s = System([Protein(rep=seq, id='EcCM', first_index=2)])
inst = s.rep_to_instance()

In [4]:
from evedesign.tools.mmseqs2 import add_sequences_mmseqs2

s = add_sequences_mmseqs2(s, use_pairing=True)

m = BoltzFoldTransformer(device='cpu', use_msa=True, diffusion_samples=1)
m.build(s)

In [5]:
output_structures = m.transform([inst])

Processing 1 inputs with 1 threads.


100%|██████████| 1/1 [00:01<00:00,  1.40s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-05-17 11:26:27.435 | INFO     | evedesign.models.boltzfold:_load_model:215 - Boltz-2 loaded from /Users/khbelahsen/.cache/boltz/boltz2_conf.ckpt
2026-05-17 11:31:05.654 | INFO     | evedesign.models.boltzfold:transform:372 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_etibfoov/predictions
2026-05-17 11:31:05.664 | INFO     | evedesign.models.boltzfold:transform:373 - Files written (6):
2026-05-17 11:31:05.668 | INFO     | evedesign.models.boltzfold:transform:376 -   instance_0/confidence_instance_0_model_0.json (443 bytes)
2026-05-17 11:31:05.669 | INFO     | evedesign.models.boltzfold:transform:376 -   instance_0/instance_0_model_0.c

In [6]:
result = output_structures[0]
print(f"Score (boltz2 confidence score): {result.score}")

Score (boltz2 confidence score): 0.9214323163032532


In [7]:
print("Confidence scores:")
for key, value in result.metadata.items():
    print(f"  {key}: {value}")
ei = result[0]
structures = ensure_sequence(ei.models["model_0"])

if ei.models:
    chain_id = structures[0].chains()[0]
    structure = structures[0]              
    print(f"\nChain: {chain_id}")
    print(f"Atom count: {len(structure.atom_array)}")
    print(f"Residue range: {structure.atom_array.res_id.min()} - {structure.atom_array.res_id.max()}")
    print(structure.atom_df().head(5))

Confidence scores:
  scores: {'model_0': {'confidence_score': 0.9214323163032532, 'ptm': 0.8522076606750488, 'iptm': 0.0, 'ligand_iptm': 0.0, 'protein_iptm': 0.0, 'complex_plddt': 0.9387384057044983, 'complex_iplddt': 0.9387384057044983, 'complex_pde': 0.3442324697971344, 'complex_ipde': 0.0, 'chains_ptm': {'0': 0.8522076606750488}, 'pair_chains_iptm': {'0': {'0': 0.8522076606750488}}}}

Chain: A
Atom count: 762
Residue range: 2 - 95
  chain_id  res_id ins_code res_name  hetero atom_name element  atom_id  \
0        A       2               THR   False         N       N        1   
1        A       2               THR   False        CA       C        2   
2        A       2               THR   False         C       C        3   
3        A       2               THR   False         O       O        4   
4        A       2               THR   False        CB       C        5   

   b_factor  occupancy  charge          x          y          z  
0    63.433        1.0       0 -20.956091 -33

In [8]:
! pip install py3Dmol -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [9]:
import io
import py3Dmol

ei = result[0]
if ei.models:
    chain_id = list(ei.models.keys())[0]
    structure = ei.models[chain_id]

    buf = io.StringIO()
    structure.to_file(buf, format="cif")
    cif_content = buf.getvalue()

    view = py3Dmol.view(width=800, height=500)
    view.addModel(cif_content, "cif")
    view.setStyle({
        "cartoon": {
            "colorscheme": {
                "prop": "b",
                "gradient": "roygb",
                "min": 50,
                "max": 90
            }
        }
    })
    view.zoomTo()
    view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.